## Homework 3

> [!NOTE]
> This homework uses the pinned 2026 lead-scoring release in the course
> repository. The plan and report are available in `cohorts/2026/data/`.

Énoncé : https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/03-classification/homework.md


### Dataset

In this homework, we will use the 2026 lead-scoring dataset. Download it from [here](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv).

Or you can do it with `wget`:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
```

In this dataset our desired target for classification task will be `converted` variable - has the client signed up to the platform or not.


In [2]:
import pandas as pd

df = pd.read_csv(
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
)

pd.set_option("display.max_columns", None)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


### Data preparation

- Check if the missing values are presented in the features.
- If there are missing values:
  - For categorical features, replace them with 'NA'
  - For numerical features, replace with with 0.0


In [3]:
print(df.isnull().sum())
categorical = list(df.select_dtypes(exclude="number").columns)
numerical = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count",
    "lead_score",
]
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)
print(df.isnull().sum())


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


### Question 1

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`


In [4]:
print(df["industry"].value_counts())

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


**Réponse Q1 : technology**

`technology` est la valeur la plus fréquente de la colonne `industry`, avec 1 173 clients sur 5 000.


### Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.


In [5]:
print(df[numerical].corr())

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  


### Split the data

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

- Make sure that the target value `converted` is not in your dataframe.


In [6]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train["converted"]
y_val = df_val["converted"]
y_test = df_test["converted"]

df_train = df_train.drop(columns="converted")
df_val = df_val.drop(columns="converted")
df_test = df_test.drop(columns="converted")

print(len(df_train), len(df_val), len(df_test))
print(df_train.columns)

3000 1000 1000
Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='str')


**Réponse Q2 : `interaction_count` et `lead_score`**

C'est la paire la plus corrélée parmi les 4 proposées, avec 0.916 : plus un client interagit, plus son score est élevé.


### Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`


In [7]:
from sklearn.metrics import mutual_info_score

mutual_info_scores = []
for col in categorical:
    mi = mutual_info_score(df_train[col], y_train)
    mutual_info_scores.append((col, round(mi, 2)))

mutual_info_scores.sort(key=lambda x: x[1], reverse=True)
print(mutual_info_scores)

[('lead_source', 0.03), ('employment_status', 0.02), ('industry', 0.0), ('location', 0.0)]


**Réponse Q3 : `lead_source`**

Calculée sur l'entraînement seulement, `lead_source` a l'information mutuelle la plus élevée avec `converted` (0.03), devant `employment_status` (0.02), `industry` (0.0) et `location` (0.0). C'est la colonne de catégories qui aide le plus à deviner si un client va s'inscrire.


### Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)
val_dict = df_val[categorical + numerical].to_dict(orient="records")
X_val = dv.transform(val_dict)
model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
print("Accuracy:", round((y_pred == y_val).mean(), 2))


Accuracy: 0.64


**Réponse Q4 : 0.65**

L'exactitude sur la validation est de 0.645, que `round` affiche 0.64 à cause de l'arrondi des nombres décimaux. Le choix le plus proche est 0.65.


### Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.


In [14]:
acc_ref = (y_pred == y_val).mean()
features = categorical + numerical
print(acc_ref)
print(features)
for f in features:
    features_subset = [x for x in features if x != f]
    train_dict = df_train[features_subset].to_dict(orient="records")
    X_train_f = dv.fit_transform(train_dict)
    val_dict = df_val[features_subset].to_dict(orient="records")
    X_val_f = dv.transform(val_dict)
    model.fit(X_train_f, y_train)
    y_pred_f = model.predict(X_val_f)
    acc = (y_pred_f == y_val).mean()
    diff = acc_ref - acc
    print(f"Accuracy without {f}: {acc}")
    print(f"{f:<26} {acc:.3f}   {diff:+.3f}")

0.645
['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
Accuracy without lead_source: 0.642
lead_source                0.642   +0.003
Accuracy without industry: 0.645
industry                   0.645   +0.000
Accuracy without employment_status: 0.645
employment_status          0.645   +0.000
Accuracy without location: 0.645
location                   0.645   +0.000
Accuracy without annual_income: 0.724
annual_income              0.724   -0.079
Accuracy without number_of_courses_viewed: 0.643
number_of_courses_viewed   0.643   +0.002
Accuracy without interaction_count: 0.601
interaction_count          0.601   +0.044
Accuracy without lead_score: 0.644
lead_score                 0.644   +0.001


**Réponse Q5 : `number_of_courses_viewed`**

Parmi les 3 colonnes proposées, c'est celle dont le retrait change le moins l'exactitude : 0.643 sans elle, contre 0.645 avec toutes les colonnes, soit une différence de 0.002. À l'inverse, `interaction_count` est la plus utile (différence de 0.044).


### Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001

> **Note**: If there are multiple options, select the smallest `C`.


In [15]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred_c = model.predict(X_val)
    print("C=%s Accuracy: %s" % (C, round((y_pred_c == y_val).mean(), 3)))

C=1e-06 Accuracy: 0.598
C=1e-05 Accuracy: 0.598
C=0.0001 Accuracy: 0.613
C=0.001 Accuracy: 0.645


**Réponse Q6 : 0.001**

Avec toutes les colonnes, `C = 0.001` donne la meilleure exactitude sur la validation (0.645). Plus `C` est petit, plus la régularisation est forte et plus l'exactitude baisse (0.598 pour `C = 0.000001`).
